## Bases vectorielles — construction, visualisation, extension

Construit les bases vectorielles SigLIP (768D, niveaux de gris) utilisées par
le RAG (`rag_generation.ipynb`, `app_rag.py`) : une base Bibles, une base
Ovide (73 illustrations thématisées `BNU_corpus.ods`), puis une base Ovide
complète (2191 illustrations, 28 éditions, métadonnées `Synthese`).

Le choix du modèle d'embedding (SigLIP gelé plutôt que CLIP/DINOv2/SigLIP
fine-tuné) a été validé séparément dans `selection_modele.ipynb` — ce
notebook recalcule ses propres embeddings via `rag_utils.charger_siglip` /
`embed_image`, pour rester exécutable seul, avec le même code que celui
utilisé au moment de la requête (retrieval).

In [ ]:
from pathlib import Path

import re
import numpy as np
import pandas as pd
import requests
import torch

from rag_utils import charger_siglip, embed_image

RACINE = Path("../../").resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)


## Bases vectorielles — Bibles + Ovide (corpus BNU·Céline)

Objectif : construire deux bases vectorielles persistantes et comparables
(SigLIP, niveaux de gris) — l'une pour les Bibles (déjà classées), l'autre
pour les illustrations d'Ovide listées par Céline dans
`retours_celine/BNU_corpus.ods` (3 feuilles : `création_1` = création du
monde, `création_2` = création de l'homme, `Déluge`).

### 1. Images Bibles et embeddings SigLIP

Dataset complet des Bibles déjà classées par thème (`classes_celine`), à
l'exclusion des classes non iconographiques (`Images CURIEUSES`,
`Images MIXTES`) — même filtre que dans `selection_modele.ipynb`.

In [ ]:
processor, model_siglip = charger_siglip(DEVICE)
print("SigLIP charge")

CLASSES_DIR = RACINE / "data" / "bibles_mdz" / "classes_celine"
THEMES_EXCLUS = {"Images CURIEUSES", "Images MIXTES"}

bible_images = []  # liste de dicts : chemin, theme
for dossier in sorted(CLASSES_DIR.iterdir()):
    if not dossier.is_dir() or dossier.name in THEMES_EXCLUS:
        continue
    fichiers = [f for f in dossier.iterdir() if f.suffix.lower() in (".jpg", ".jpeg", ".png")]
    for f in fichiers:
        bible_images.append({"chemin": f, "theme": dossier.name})

themes_bible = [i["theme"] for i in bible_images]
print(f"{len(bible_images)} images Bible sur {len(set(themes_bible))} themes")

chemins_bible = [i["chemin"] for i in bible_images]
print("Embeddings SigLIP (Bibles)...")
X_siglip_bible = np.array([
    embed_image(chemin, processor, model_siglip, DEVICE) for chemin in chemins_bible
])
print(X_siglip_bible.shape)


### 2. Tableau unifié BNU_corpus (3 feuilles -> format long)

In [ ]:
import pandas as pd
import requests

def nettoyer_feuille(df, theme):
    df = df.copy()
    df = df[df['titre'].notna()]
    df = df[df['titre'].astype(str).str.strip().str.lower() != 'titre']  # ligne d'en-tete parasite
    for col in ['url_1', 'url_2']:
        if col in df.columns:
            df[col] = df[col].replace({'absent': np.nan, 'Absent': np.nan})
    df['theme'] = theme
    return df

COLS_20 = ['url_1','url_2','url_catalogue','url_gallica','type','titre','ville','publisher','annee','langue',
           'format','nb_pages','nb_vues_num','nb_gravures','technique','graveur','taille_gravures',
           'type_iconographique','cadre','autres']

CHEMIN_BNU_CORPUS = RACINE / "retours_celine" / "BNU_corpus.ods"

c1 = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='création_1', header=None)
c1.columns = COLS_20
c1 = nettoyer_feuille(c1, 'creation_monde')

c2 = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='création_2')
c2.columns = ['url_1','url_catalogue','url_gallica','type','titre','ville','publisher','annee','langue',
              'format','nb_pages','nb_vues_num','nb_gravures','technique','graveur','taille_gravures',
              'type_iconographique','cadre','autres']
c2['url_2'] = np.nan
c2 = nettoyer_feuille(c2, 'creation_homme')

dl = pd.read_excel(CHEMIN_BNU_CORPUS, engine='odf', sheet_name='Déluge')
dl.columns = COLS_20
dl = nettoyer_feuille(dl, 'deluge')

bnu_corpus = pd.concat([c1, c2, dl], ignore_index=True)
bnu_corpus['annee'] = pd.to_numeric(bnu_corpus['annee'], errors='coerce')
bnu_corpus['variante'] = 'principale'

# url_2, quand renseignee, pointe vers une AUTRE illustration de la meme edition
# et du meme theme (verifie : jamais identique a url_1) - on l'ajoute comme ligne
# supplementaire plutot que de l'ignorer.
url2_valides = bnu_corpus[
    bnu_corpus['url_2'].notna() & (bnu_corpus['url_2'].astype(str) != bnu_corpus['url_1'].astype(str))
].copy()
url2_valides['url_1'] = url2_valides['url_2']
url2_valides['variante'] = 'secondaire'

bnu_corpus = pd.concat([bnu_corpus, url2_valides], ignore_index=True)
print(f"{len(bnu_corpus)} lignes ({len(url2_valides)} variantes url_2 ajoutees)")
print(bnu_corpus['theme'].value_counts())

### 3. Résolution des images

Stratégie en deux temps :
1. **Réutiliser l'existant** — 20 des éditions listées dans BNU_corpus ont déjà
   été téléchargées et segmentées par ailleurs dans ce projet (corpus de 28
   éditions dans `data/editions_ovide/segmentees/`, construit pour la
   classification bois/cuivre). On y cherche directement la page ciblée avant
   de retélécharger quoi que ce soit.
2. **Téléchargement + découpe YOLO** — pour les pages non couvertes par
   l'existant, on télécharge la page via l'API IIIF (Gallica ou
   digitale-sammlungen) et on applique le même détecteur YOLO que pour le
   reste du corpus Ovide (`gallica_utils.segmenter_page`), pour rester
   méthodologiquement cohérent avec le corpus déjà construit.

In [24]:
import os

# éditions BNU_corpus déjà téléchargées + segmentées ailleurs dans le projet
# (classification_graveur/01_constitution_dataset.ipynb, classification_bois_cuivre/01_dataset.ipynb)
ID_VERS_DOSSIER = {
    'btv1b2200047r':  'bois_salomon_rouille_lyon1557',
    'bsb10139926':    'bois_wickram_behem_mayence1545',
    'bsb00087854':    'bois_solis_feyerabend_francfort1581',
    'bsb10863401':    'cuivre_savery_farnaby_paris1637',
    'btv1b22000559':  'bois_eskrich_rouille_lyon1556',
    'bsb11054210':    'bois_leroy_gueynard_lyon1510',
    'bsb10872075':    'cuivre_baur_sn_augsbourg1709',
    'bsb10872073':    'cuivre_baur_sn_vienne1639',
    'bsb00004340':    'cuivre_borcht_plantin_anvers1591',
    'bpt6k15151988':  'cuivre_bouche_blaeu_amsterdam1702',
    'bpt6k15218623':  'cuivre_depasse_depasse_koln1602',
    'bpt6k1522448r':  'cuivre_depasse_jansonius_arnhem1607',
    'bsb11913284':    'cuivre_gaultier_guillemot_paris1610',
    'bpt6k6277348n':  'cuivre_gaultier_veuveguillemot_paris1614',
    'btv1b10534945n': 'cuivre_goltzius_goltzius_haarlem1589',
    'bpt6k722055':    'cuivre_isaac_langelier_paris1617',
    'btv1b22000826':  'cuivre_mathieu_langelier_paris1619',
    'bpt6k87045023':  'cuivre_monconet_sommaville_paris1660',
    'btv1b54000051z': 'cuivre_tempesta_dejode_anvers1606',
    'bsb00008186':    'cuivre_tempesta_jansonius_amsterdam1610',
}

SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

def indexer_dossier(dossier):
    """Indexe {page: [(confiance, nom_fichier), ...]} pour les crops déjà présents."""
    idx = {}
    for f in os.listdir(SEG_DIR / dossier):
        m = re.search(r'0*(\d+)_det(\d+)_conf([\d.]+)\.jpg$', f)
        if m:
            page, conf = int(m.group(1)), float(m.group(3))
            idx.setdefault(page, []).append((conf, f))
    return idx

INDEX_LOCAL = {d: indexer_dossier(d) for d in set(ID_VERS_DOSSIER.values())}

PAT_GALLICA = re.compile(r'ark:/12148/([a-zA-Z0-9]+)/f(\d+)')
PAT_MDZ = re.compile(r'digitale-sammlungen\.de/en/view/(bsb\d+)\?page=(\d+)')

def resoudre_source(url):
    """Retourne (statut, source, identifiant, page, chemin_local_existant|None)."""
    if pd.isna(url):
        return 'sans_url', None, None, None, None
    url = str(url)
    m = PAT_GALLICA.search(url)
    source = 'gallica'
    if not m:
        m = PAT_MDZ.search(url)
        source = 'mdz'
    if not m:
        return 'ignore', None, None, None, None  # hathitrust ou format non gere

    ident, page = m.group(1), int(m.group(2))
    dossier = ID_VERS_DOSSIER.get(ident)
    if dossier:
        idx = INDEX_LOCAL[dossier]
        if page in idx:
            conf, fichier = max(idx[page], key=lambda t: t[0])
            return 'local', source, ident, page, str(SEG_DIR / dossier / fichier)
        proches = [p for p in idx if abs(p - page) <= 2]
        if proches:
            p_best = min(proches, key=lambda p: abs(p - page))
            conf, fichier = max(idx[p_best], key=lambda t: t[0])
            return 'local', source, ident, page, str(SEG_DIR / dossier / fichier)

    return 'a_telecharger', source, ident, page, None

resolution = bnu_corpus['url_1'].apply(resoudre_source)
bnu_corpus[['statut', 'source', 'ident', 'page', 'chemin_local']] = pd.DataFrame(
    resolution.tolist(), index=bnu_corpus.index
)
print(bnu_corpus['statut'].value_counts())
print(f"\n{(bnu_corpus['statut'] == 'local').sum()} pages déjà disponibles localement (corpus déjà segmenté)")

statut
a_telecharger    70
local            21
sans_url          3
ignore            1
Name: count, dtype: int64

21 pages déjà disponibles localement (corpus déjà segmenté)


Téléchargement + découpe YOLO pour les pages non couvertes par l'existant.

In [25]:
import sys
import time

sys.path.append(str(RACINE / "notebooks"))
from gallica_utils import charger_yolo, segmenter_page, liberer_yolo

DOSSIER_PAGES_BNU = RACINE / "data" / "editions_ovide" / "pages_brutes" / "bnu_corpus_celine"
DOSSIER_CROPS_BNU = RACINE / "data" / "editions_ovide" / "segmentees" / "bnu_corpus_celine"
DOSSIER_PAGES_BNU.mkdir(parents=True, exist_ok=True)
DOSSIER_CROPS_BNU.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Gecko/20100101 Firefox/128.0"}

def url_page_brute(source, ident, page):
    if source == 'gallica':
        return f"https://gallica.bnf.fr/iiif/ark:/12148/{ident}/f{page}/full/full/0/native.jpg"
    return f"https://api.digitale-sammlungen.de/iiif/image/v2/{ident}_{page:05d}/full/full/0/default.jpg"

def telecharger_page(source, ident, page, max_essais=4, pause_base=3):
    """Téléchargement avec pause entre requêtes + repli exponentiel sur erreur
    réseau/429 (Gallica bloque les rafales de requêtes sans délai)."""
    page = int(page)
    chemin_page = DOSSIER_PAGES_BNU / f"{ident}_f{page:03d}.jpg"
    if chemin_page.exists():
        return chemin_page, None

    derniere_erreur = None
    for essai in range(max_essais):
        try:
            r = requests.get(url_page_brute(source, ident, page), headers=HEADERS, timeout=30)
            if r.status_code == 429:
                attente = pause_base * (2 ** essai)
                derniere_erreur = f"429 persistant (essai {essai + 1})"
                time.sleep(attente)
                continue
            r.raise_for_status()
            chemin_page.write_bytes(r.content)
            return chemin_page, None
        except Exception as e:
            derniere_erreur = str(e)
            time.sleep(pause_base * (2 ** essai))
    return None, derniere_erreur

a_telecharger = bnu_corpus[bnu_corpus['statut'].isin(['a_telecharger', 'erreur_telechargement', 'aucune_illustration_detectee'])]
print(f"{len(a_telecharger)} pages à télécharger + segmenter")

modele_yolo = charger_yolo(str(RACINE / "yolov5_repo"))

resultats_dl = {}
erreurs_detail = {}
for i, (idx, row) in enumerate(a_telecharger.iterrows(), 1):
    print(f"  {i}/{len(a_telecharger)}...", end="\r")
    cle = (row['source'], row['ident'], row['page'])
    if cle in resultats_dl:
        continue
    chemin_page, erreur = telecharger_page(*cle)
    time.sleep(1.0 if row['source'] == 'gallica' else 0.2)  # pacing anti rate-limit
    if erreur:
        resultats_dl[cle] = ('erreur_telechargement', None)
        erreurs_detail[cle] = erreur
        continue
    prefixe = f"{row['ident']}_f{int(row['page']):03d}"
    nb = segmenter_page(str(chemin_page), prefixe, str(DOSSIER_CROPS_BNU), modele_yolo, conf_thres=0.25)
    if nb == 0:
        resultats_dl[cle] = ('aucune_illustration_detectee', None)
        continue
    crops = sorted(DOSSIER_CROPS_BNU.glob(f"{prefixe}_det*_conf*.jpg"))
    meilleur = max(crops, key=lambda p: float(re.search(r'conf([\d.]+)\.jpg$', p.name).group(1)))
    resultats_dl[cle] = ('telecharge', str(meilleur))
print()

liberer_yolo(modele_yolo)

for idx, row in a_telecharger.iterrows():
    statut, chemin = resultats_dl[(row['source'], row['ident'], row['page'])]
    bnu_corpus.loc[idx, 'statut'] = statut
    if chemin:
        bnu_corpus.loc[idx, 'chemin_local'] = chemin

print(bnu_corpus['statut'].value_counts())
if erreurs_detail:
    print("\nExemples d'erreurs :")
    for cle, msg in list(erreurs_detail.items())[:5]:
        print(f"  {cle} : {msg}")

70 pages à télécharger + segmenter


✓ YOLO chargé — classes : {0: 'illustration'}



✓ Mémoire GPU libérée
statut
telecharge                      61
local                           21
aucune_illustration_detectee     9
sans_url                         3
ignore                           1
Name: count, dtype: int64


### 4. Vectorisation SigLIP (niveaux de gris) des illustrations Ovide résolues

In [ ]:
ovide_bnu_resolus = bnu_corpus[bnu_corpus['chemin_local'].notna()].copy()
avant_dedup = len(ovide_bnu_resolus)
ovide_bnu_resolus = ovide_bnu_resolus.drop_duplicates(subset='chemin_local').reset_index(drop=True)
print(f"{len(ovide_bnu_resolus)} illustrations Ovide (BNU_corpus) résolues sur {len(bnu_corpus)} lignes"
      f" ({avant_dedup - len(ovide_bnu_resolus)} doublons de fichier retires)")
print(ovide_bnu_resolus['theme'].value_counts())
print(ovide_bnu_resolus['variante'].value_counts())

chemins_ovide_bnu = [Path(p) for p in ovide_bnu_resolus['chemin_local']]
print("\nEmbeddings SigLIP (Ovide BNU_corpus)...")
X_siglip_ovide_bnu = np.array([embed_image(chemin, processor, model_siglip, DEVICE) for chemin in chemins_ovide_bnu])
print(X_siglip_ovide_bnu.shape)

### 5. Sauvegarde des deux bases vectorielles

Format : un fichier pickle par base (pas de dépendance `pyarrow`/`fastparquet`
installée dans l'environnement) — une ligne par illustration, avec son chemin
local, son thème / ses métadonnées d'édition, et son embedding SigLIP (768D).

In [27]:
DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
DOSSIER_VECTOR_DB.mkdir(exist_ok=True)

# URLs distantes (Gallica / digitale-sammlungen) plutôt que des chemins locaux,
# pour que le fichier reste utilisable si on le partage (ex: HTML envoyé à un
# collègue qui n'a pas les images sur son disque).
def urls_bsb(bsb_id, page):
    page = int(page)
    return (
        f"https://www.digitale-sammlungen.de/en/view/{bsb_id}?page={page}",
        f"https://api.digitale-sammlungen.de/iiif/image/v2/{bsb_id}_{page:05d}/full/full/0/default.jpg",
    )

def urls_gallica(ark, folio):
    folio = int(folio)
    return (
        f"https://gallica.bnf.fr/ark:/12148/{ark}/f{folio}.item",
        f"https://gallica.bnf.fr/iiif/ark:/12148/{ark}/f{folio}/full/full/0/native.jpg",
    )

# Base vectorielle Bibles
base_bibles = pd.DataFrame({
    "chemin": [str(i["chemin"]) for i in bible_images],
    "theme": themes_bible,
})
base_bibles["embedding"] = [v.tolist() for v in X_siglip_bible]

# le bsb_id et le numero de page sont toujours presents dans le nom de fichier
# (ex: bsb00085642_page031_det1_conf0.92.jpg) - tout le corpus Bibles vient de
# digitale-sammlungen (dossier data/bibles_mdz/).
PAT_BIBLE = re.compile(r'^(bsb\d+)_page(\d+)_det')
url_page_b, url_image_b = [], []
for chemin in base_bibles["chemin"]:
    m = PAT_BIBLE.match(Path(chemin).name)
    up, ui = urls_bsb(m.group(1), m.group(2)) if m else (None, None)
    url_page_b.append(up)
    url_image_b.append(ui)
base_bibles["url_page"] = url_page_b
base_bibles["url_image"] = url_image_b

base_bibles.to_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
print(f"Base vectorielle Bibles : {base_bibles.shape} -> {DOSSIER_VECTOR_DB / 'bibles_siglip.pkl'}")
print(f"  url_page manquante : {base_bibles['url_page'].isna().sum()} / {len(base_bibles)}")

# Base vectorielle Ovide (corpus BNU . Céline)
base_ovide = ovide_bnu_resolus[[
    "chemin_local", "theme", "titre", "ville", "publisher", "annee", "langue", "graveur", "technique", "variante",
    "ident", "page", "source"
]].rename(columns={"chemin_local": "chemin", "source": "source_plateforme"})
base_ovide["embedding"] = [v.tolist() for v in X_siglip_ovide_bnu]

url_page_o, url_image_o = [], []
for _, r in base_ovide.iterrows():
    if r["source_plateforme"] == "gallica":
        up, ui = urls_gallica(r["ident"], r["page"])
    else:
        up, ui = urls_bsb(r["ident"], r["page"])
    url_page_o.append(up)
    url_image_o.append(ui)
base_ovide["url_page"] = url_page_o
base_ovide["url_image"] = url_image_o

print(f"Base vectorielle Ovide (73 illustrations, en memoire) : {base_ovide.shape}")

Base vectorielle Bibles : (447, 5) -> /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/bibles_siglip.pkl
  url_page manquante : 0 / 447
Base vectorielle Ovide (73 illustrations, en memoire) : (73, 16)


## Visualisation de proximité — Bibles + Ovide

Un même espace de points : chaque illustration (Bible ou Ovide) est placée par
projection UMAP de son embedding SigLIP 768D. La **couleur** encode le thème
(regroupé sur une taxonomie commune aux deux corpus), la **forme** distingue
Bible (cercle) et Ovide (triangle). Des liens fins relient chaque illustration à
ses plus proches voisines (similarité cosinus dans l'espace 768D d'origine, pas
dans la projection 2D) — leur opacité est proportionnelle à la similarité.

### 1. Chargement et taxonomie commune (regroupement des thèmes Bibles/Ovide)

In [28]:
import umap

base_bibles = pd.read_pickle(DOSSIER_VECTOR_DB / "bibles_siglip.pkl")
# base_ovide deja calcule ci-dessus (plus jamais sauvegarde sur disque a part)

# Taxonomie commune : les 12 classes fines des Bibles sont regroupées sous les 3
# thèmes partagés avec Ovide ; les classes sans équivalent Ovide restent groupées
# à part ("autre_bible") pour ne pas être comparées à tort.
MAPPING_THEME_BIBLE = {
    "Creation du MONDE": "creation_monde",
    "Creation HOMME": "creation_homme",
    "Creation EVE": "creation_homme",
    "Creation HORS ENTRAINEMENT": "creation_homme",
    "Deluge": "deluge",
    "Deluge AVANT": "deluge",
    "Deluge APRES": "deluge",
    "Babel": "autre_bible",
    "Cain et Abel": "autre_bible",
    "Chasses du paradis": "autre_bible",
    "Sacrifice ABRAHAM": "autre_bible",
    "Tentation": "autre_bible",
}

base_bibles = base_bibles.copy()
base_bibles["source"] = "bible"
base_bibles["theme_groupe"] = base_bibles["theme"].map(MAPPING_THEME_BIBLE)
base_bibles["theme_detail"] = base_bibles["theme"]
base_bibles["titre"] = np.nan  # pas de titre distinct du theme pour les Bibles

base_ovide = base_ovide.copy()
base_ovide["source"] = "ovide"
base_ovide["theme_groupe"] = base_ovide["theme"]
base_ovide["theme_detail"] = base_ovide["theme"]

points = pd.concat([
    base_bibles[["chemin", "source", "theme_groupe", "theme_detail", "titre", "embedding", "url_page", "url_image"]],
    base_ovide[["chemin", "source", "theme_groupe", "theme_detail", "titre", "embedding", "url_page", "url_image"]],
], ignore_index=True)

X_points = np.array(points["embedding"].tolist())
print(f"{len(points)} points -> embeddings {X_points.shape}")
print(points.groupby(["source", "theme_groupe"]).size())

520 points -> embeddings (520, 768)
source  theme_groupe  
bible   autre_bible       222
        creation_homme    107
        deluge             89
ovide   creation_homme     17
        creation_monde     25
        deluge             31
dtype: int64


### 2. Projection 2D (UMAP) et liens de plus proches voisins

In [29]:
# Projection 2D (UMAP, distance cosinus) pour la disposition visuelle
reducteur = umap.UMAP(n_neighbors=15, min_dist=0.3, metric="cosine", random_state=42)
coords = reducteur.fit_transform(X_points)
points["x"] = coords[:, 0]
points["y"] = coords[:, 1]

# Liens vers les plus proches voisins, calculés dans l'espace d'origine (768D),
# pas dans la projection 2D (qui déforme les distances).
sim_points = cosine_similarity(X_points)
np.fill_diagonal(sim_points, -1)

K_VOISINS = 3  # utilisé seulement pour la vérification trustworthiness plus bas
SEUIL_SIMILARITE = 0.95

# Un lien par paire dont la similarité dépasse le seuil (plus de plafond à k
# voisins par point : un point très central peut avoir beaucoup de liens, un
# point isolé peut n'en avoir aucun).
i_idx, j_idx = np.triu_indices_from(sim_points, k=1)
sims_paires = sim_points[i_idx, j_idx]
garder = sims_paires >= SEUIL_SIMILARITE

liens = [
    {"source": int(i), "target": int(j), "similarite": float(s)}
    for i, j, s in zip(i_idx[garder], j_idx[garder], sims_paires[garder])
]

print(f"{len(liens)} liens (seuil de similarité >= {SEUIL_SIMILARITE}, toutes les paires, sans plafond par point)")

488 liens (seuil de similarité >= 0.95, toutes les paires, sans plafond par point)


### 3. Correspondances Ovide -> Bible (le thème est-il retrouvé ?)

In [30]:
# Correspondances Ovide -> Bible : pour chaque illustration Ovide, sa MEILLEURE
# correspondance Bible (n'importe quel theme), independamment du graphe de
# voisinage local ci-dessus. Sert a mesurer si l'iconographie Ovide "retombe"
# bien sur le meme theme cote Bible, ou sur un theme different.
ovide_idx = points.index[points["source"] == "ovide"].to_numpy()
bible_idx = points.index[points["source"] == "bible"].to_numpy()

X_ovide = X_points[ovide_idx]
X_bible = X_points[bible_idx]
sim_croisee = cosine_similarity(X_ovide, X_bible)

correspondances = []
for i, idx_o in enumerate(ovide_idx):
    j_meilleur = int(sim_croisee[i].argmax())
    idx_b = int(bible_idx[j_meilleur])
    correspondances.append({
        "ovide_id": int(idx_o),
        "bible_id": idx_b,
        "similarite": float(sim_croisee[i, j_meilleur]),
        "meme_theme": bool(points.loc[idx_o, "theme_groupe"] == points.loc[idx_b, "theme_groupe"]),
    })

df_correspondances = pd.DataFrame(correspondances)
df_correspondances["theme_ovide"] = df_correspondances["ovide_id"].map(points["theme_groupe"])

stats_theme = df_correspondances.groupby("theme_ovide").agg(
    n=("similarite", "size"),
    pct_meme_theme=("meme_theme", "mean"),
    similarite_moyenne=("similarite", "mean"),
).reset_index()

print("Correspondance Ovide -> meilleure Bible (n'importe quel theme), par theme Ovide :")
for _, r in stats_theme.iterrows():
    print(f"  {r['theme_ovide']:16s} : n={int(r['n']):2d}  meme theme={r['pct_meme_theme']:.0%}  similarite moyenne={r['similarite_moyenne']:.3f}")

# Correspondances Bible -> Ovide (sens inverse) : pour chaque illustration
# Bible, sa MEILLEURE correspondance Ovide - meme raisonnement, direction opposee.
correspondances_bible = []
for j, idx_b in enumerate(bible_idx):
    i_meilleur = int(sim_croisee[:, j].argmax())
    idx_o = int(ovide_idx[i_meilleur])
    correspondances_bible.append({
        "bible_id": int(idx_b),
        "ovide_id": idx_o,
        "similarite": float(sim_croisee[i_meilleur, j]),
        "meme_theme": bool(points.loc[idx_b, "theme_groupe"] == points.loc[idx_o, "theme_groupe"]),
    })

df_correspondances_bible = pd.DataFrame(correspondances_bible)
df_correspondances_bible["theme_bible"] = df_correspondances_bible["bible_id"].map(points["theme_groupe"])

# stats seulement sur les 3 themes partages ("autre_bible" n'a pas d'equivalent Ovide,
# la comparaison n'aurait pas de sens)
stats_theme_bible = (
    df_correspondances_bible[df_correspondances_bible["theme_bible"] != "autre_bible"]
    .groupby("theme_bible")
    .agg(
        n=("similarite", "size"),
        pct_meme_theme=("meme_theme", "mean"),
        similarite_moyenne=("similarite", "mean"),
    ).reset_index()
)

print("\nCorrespondance Bible -> meilleure Ovide (n'importe quel theme), par theme Bible (themes partages seulement) :")
for _, r in stats_theme_bible.iterrows():
    print(f"  {r['theme_bible']:16s} : n={int(r['n']):3d}  meme theme={r['pct_meme_theme']:.0%}  similarite moyenne={r['similarite_moyenne']:.3f}")

Correspondance Ovide -> meilleure Bible (n'importe quel theme), par theme Ovide :
  creation_homme   : n=17  meme theme=76%  similarite moyenne=0.899
  creation_monde   : n=25  meme theme=0%  similarite moyenne=0.879
  deluge           : n=31  meme theme=23%  similarite moyenne=0.893

Correspondance Bible -> meilleure Ovide (n'importe quel theme), par theme Bible (themes partages seulement) :
  creation_homme   : n=107  meme theme=53%  similarite moyenne=0.879
  deluge           : n= 89  meme theme=63%  similarite moyenne=0.851


### 4. Export du HTML interactif (données + gabarit)

In [31]:
import json as jsonlib

noeuds_json = []
for i, row in points.iterrows():
    noeuds_json.append({
        "id": int(i),
        "x": float(row["x"]),
        "y": float(row["y"]),
        "source": row["source"],
        "theme": row["theme_groupe"],
        "detail": row["theme_detail"],
        "titre": (str(row["titre"])[:90] if pd.notna(row["titre"]) else ""),
        "img": (row["url_image"] if pd.notna(row["url_image"]) else ""),
        "source_url": (row["url_page"] if pd.notna(row["url_page"]) else ""),
    })

stats_json = [
    {
        "theme": str(r["theme_ovide"]),
        "n": int(r["n"]),
        "pct_meme_theme": float(r["pct_meme_theme"]),
        "similarite_moyenne": float(r["similarite_moyenne"]),
    }
    for _, r in stats_theme.iterrows()
]

stats_bible_json = [
    {
        "theme": str(r["theme_bible"]),
        "n": int(r["n"]),
        "pct_meme_theme": float(r["pct_meme_theme"]),
        "similarite_moyenne": float(r["similarite_moyenne"]),
    }
    for _, r in stats_theme_bible.iterrows()
]

DONNEES_JSON = jsonlib.dumps({
    "noeuds": noeuds_json,
    "liens": liens,
    "correspondances": correspondances,
    "correspondances_bible": correspondances_bible,
    "stats": stats_json,
    "stats_bible": stats_bible_json,
}, ensure_ascii=False)
print(f"{len(noeuds_json)} noeuds, {len(liens)} liens, {len(correspondances)} correspondances Ovide->Bible, "
      f"{len(correspondances_bible)} correspondances Bible->Ovide -> JSON pret ({len(DONNEES_JSON)/1024:.0f} Ko)")

520 noeuds, 488 liens, 73 correspondances Ovide->Bible, 447 correspondances Bible->Ovide -> JSON pret (247 Ko)


In [32]:
HTML_TEMPLATE = """<!doctype html>
<html lang="fr">
<head>
<meta charset="utf-8">
<title>Proximite Bibles / Ovide - SigLIP</title>
<style>
  .viz-root {
    color-scheme: light;
    --surface-1:      #fcfcfb;
    --page-plane:     #f9f9f7;
    --text-primary:   #0b0b0b;
    --text-secondary: #52514e;
    --text-muted:     #898781;
    --gridline:       #e1e0d9;
    --border:         rgba(11,11,11,0.10);
    --series-1: #2a78d6; /* creation_monde */
    --series-2: #008300; /* creation_homme */
    --series-3: #e87ba4; /* deluge */
    --series-4: #eda100; /* autre_bible */
  }
  @media (prefers-color-scheme: dark) {
    :root:where(:not([data-theme="light"])) .viz-root {
      color-scheme: dark;
      --surface-1:      #1a1a19;
      --page-plane:     #0d0d0d;
      --text-primary:   #ffffff;
      --text-secondary: #c3c2b7;
      --text-muted:     #898781;
      --gridline:       #2c2c2a;
      --border:         rgba(255,255,255,0.10);
      --series-1: #3987e5;
      --series-2: #008300;
      --series-3: #d55181;
      --series-4: #c98500;
    }
  }
  :root[data-theme="dark"] .viz-root {
    color-scheme: dark;
    --surface-1:      #1a1a19;
    --page-plane:     #0d0d0d;
    --text-primary:   #ffffff;
    --text-secondary: #c3c2b7;
    --text-muted:     #898781;
    --gridline:       #2c2c2a;
    --border:         rgba(255,255,255,0.10);
    --series-1: #3987e5;
    --series-2: #008300;
    --series-3: #d55181;
    --series-4: #c98500;
  }
  * { box-sizing: border-box; }
  html, body { margin: 0; padding: 0; overflow-x: hidden; }
  body {
    font-family: system-ui, -apple-system, "Segoe UI", sans-serif;
    background: var(--page-plane);
    color: var(--text-primary);
  }
  .viz-root { padding: 20px; }
  h1 { font-size: 1.15em; margin: 0 0 4px; }
  .sous-titre { color: var(--text-secondary); font-size: 0.85em; margin: 0 0 16px; }
  .cadre {
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 8px;
    position: relative;
    width: 100%;
    height: 78vh;
    min-height: 480px;
    overflow: hidden;
  }
  svg { width: 100%; height: 100%; display: block; cursor: grab; }
  svg:active { cursor: grabbing; }
  .lien { stroke: var(--text-muted); fill: none; }
  .marque { stroke: var(--border); stroke-width: 1px; cursor: pointer; }
  .marque.estompe { opacity: 0.08; }
  .legende {
    position: absolute; top: 12px; right: 12px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 10px 12px;
    font-size: 0.78em;
    color: var(--text-secondary);
    max-width: 220px;
  }
  .legende h4 { margin: 0 0 6px; font-size: 0.9em; color: var(--text-primary); font-weight: 600; }
  .legende-item { display: flex; align-items: center; gap: 6px; margin: 3px 0; cursor: pointer; user-select: none; }
  .legende-item.inactif { opacity: 0.35; }
  .legende-puce { width: 10px; height: 10px; border-radius: 50%; flex: none; }
  .legende-sep { height: 1px; background: var(--gridline); margin: 8px 0; }
  .legende-forme { width: 10px; height: 10px; flex: none; }
  .infobulle {
    position: absolute;
    pointer-events: none;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 8px;
    font-size: 0.78em;
    color: var(--text-primary);
    box-shadow: 0 4px 16px rgba(0,0,0,0.18);
    max-width: 200px;
    opacity: 0;
    transition: opacity 0.1s;
    z-index: 10;
  }
  .infobulle img { width: 100%; border-radius: 4px; margin-bottom: 6px; display: block; }
  .infobulle .theme { color: var(--text-secondary); }
  .infobulle.epingle { pointer-events: auto; }
  .infobulle .lien-voir {
    display: inline-block;
    margin-top: 6px;
    color: var(--text-primary);
    text-decoration: underline;
    font-weight: 600;
  }
  .infobulle .fermer-infobulle { margin-top: 6px; color: var(--text-muted); font-size: 0.9em; }
  .aide { color: var(--text-muted); font-size: 0.75em; margin-top: 8px; }
  .bascule {
    position: absolute; top: 12px; left: 12px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 6px;
    padding: 4px;
    display: flex; gap: 4px;
    font-size: 0.78em;
  }
  .bascule button {
    font: inherit;
    border: none;
    background: transparent;
    color: var(--text-secondary);
    padding: 5px 9px;
    border-radius: 4px;
    cursor: pointer;
  }
  .bascule button.actif { background: var(--gridline); color: var(--text-primary); font-weight: 600; }
  .lien-correspondance.autreTheme { stroke-dasharray: 3,3; }
  .panneau-stats {
    margin-top: 14px;
    background: var(--surface-1);
    border: 1px solid var(--border);
    border-radius: 8px;
    padding: 12px 16px;
  }
  .panneau-stats h3 { margin: 0 0 4px; font-size: 0.95em; }
  .panneau-stats p.note { color: var(--text-secondary); font-size: 0.78em; margin: 0 0 10px; }
  .panneau-stats table { border-collapse: collapse; width: 100%; font-size: 0.82em; }
  .panneau-stats th, .panneau-stats td { text-align: left; padding: 5px 10px; border-bottom: 1px solid var(--gridline); }
  .panneau-stats th { color: var(--text-muted); font-weight: 500; }
  .panneau-stats td.chiffre { font-variant-numeric: tabular-nums; }
</style>
</head>
<body>
<div class="viz-root">
  <h1>Proximite des illustrations - Bibles (classes_celine) vs Ovide (BNU_corpus)</h1>
  <p class="sous-titre">__N_NOEUDS__ illustrations, embeddings SigLIP (niveaux de gris) projetes en 2D (UMAP, distance cosinus). Cliquer la legende pour filtrer. Molette = zoom, glisser = deplacer.</p>
  <div class="cadre">
    <svg></svg>
    <div class="bascule" id="bascule">
      <button data-mode="local" class="actif">Voisinage local</button>
      <button data-mode="correspondance">Correspondances Ovide -&gt; Bible</button>
      <button data-mode="correspondance_bible">Correspondances Bible -&gt; Ovide</button>
    </div>
    <div class="legende" id="legende"></div>
    <div class="infobulle" id="infobulle"></div>
  </div>
  <p class="aide" id="aide-liens"></p>

  <div class="panneau-stats">
    <h3>Ovide -&gt; meilleure correspondance Bible, par theme</h3>
    <p class="note">Pour chaque illustration Ovide, on cherche l'illustration Bible la plus
    proche (n'importe quel theme). "% meme theme" = a quelle frequence cette meilleure
    correspondance porte deja le meme theme cote Bible.</p>
    <table id="table-stats"></table>

    <h3 style="margin-top:16px;">Bible -&gt; meilleure correspondance Ovide, par theme</h3>
    <p class="note">Sens inverse : pour chaque illustration Bible d'un des 3 themes partages
    avec Ovide, sa meilleure correspondance Ovide. "Autre theme biblique" est exclu ici -
    Ovide n'a pas d'equivalent pour ces themes.</p>
    <table id="table-stats-bible"></table>
  </div>
</div>

<script src="https://unpkg.com/d3@7/dist/d3.min.js"></script>
<script>
const DONNEES = __DONNEES_JSON__;

const THEMES = [
  { cle: "creation_monde", label: "Creation du monde", var: "--series-1" },
  { cle: "creation_homme", label: "Creation de l'homme / Eve", var: "--series-2" },
  { cle: "deluge",         label: "Deluge",                     var: "--series-3" },
  { cle: "autre_bible",    label: "Autre theme biblique",       var: "--series-4" },
];

const racine = document.querySelector(".viz-root");
const style = getComputedStyle(racine);
const couleurTheme = {};
THEMES.forEach(t => couleurTheme[t.cle] = style.getPropertyValue(t.var).trim());

const cadre = document.querySelector(".cadre");
const svg = d3.select("svg");
const g = svg.append("g");
const gLiens = g.append("g").attr("class", "couche-liens");
const gNoeuds = g.append("g").attr("class", "couche-noeuds");

const infobulle = d3.select("#infobulle");
let infobullePinglee = false;

function contenuInfobulle(d) {
  return `
    <img src="${d.img}" loading="lazy" onerror="this.style.display='none'">
    <div><b>${d.source === "bible" ? "Bible" : "Ovide"}</b></div>
    <div class="theme">${d.detail}</div>
    ${d.titre ? `<div class="theme">${d.titre}</div>` : ""}
    ${d.source_url ? `<a class="lien-voir" href="${d.source_url}" target="_blank" rel="noopener">Voir la page source &#8599;</a>` : ""}
  `;
}

svg.on("click", () => {
  if (!infobullePinglee) return;
  infobullePinglee = false;
  infobulle.classed("epingle", false).style("opacity", 0);
});

let largeur = cadre.clientWidth, hauteur = cadre.clientHeight;
let xEchelleActuelle = null, yEchelleActuelle = null;
// declares tot : dessiner() (appele plus bas) invoque dessinerLiens(), qui en depend
let modeLiens = "local";
const parIdIndex = new Map(DONNEES.noeuds.map(n => [n.id, n]));
const simCorr = d3.extent(DONNEES.correspondances, d => d.similarite);
const simCorrBible = d3.extent(DONNEES.correspondances_bible, d => d.similarite);
const simLocal = d3.extent(DONNEES.liens, d => d.similarite);

const formeParSource = {
  bible: d3.symbol().type(d3.symbolCircle).size(64),
  ovide: d3.symbol().type(d3.symbolTriangle).size(64),
};

function dessiner() {
  largeur = cadre.clientWidth;
  hauteur = cadre.clientHeight;
  svg.attr("viewBox", [0, 0, largeur, hauteur]);

  const xExtent = d3.extent(DONNEES.noeuds, d => d.x);
  const yExtent = d3.extent(DONNEES.noeuds, d => d.y);
  const marge = 40;
  const xScale = d3.scaleLinear().domain(xExtent).range([marge, largeur - marge]);
  const yScale = d3.scaleLinear().domain(yExtent).range([hauteur - marge, marge]);
  xEchelleActuelle = xScale;
  yEchelleActuelle = yScale;

  dessinerLiens();

  gNoeuds.selectAll("path")
    .data(DONNEES.noeuds)
    .join("path")
    .attr("class", "marque")
    .attr("transform", d => `translate(${xScale(d.x)},${yScale(d.y)})`)
    .attr("d", d => formeParSource[d.source]())
    .attr("fill", d => couleurTheme[d.theme] || "#999")
    .on("mouseenter", (evt, d) => {
      if (infobullePinglee) return;
      infobulle.style("opacity", 1).html(contenuInfobulle(d));
    })
    .on("mousemove", (evt) => {
      if (infobullePinglee) return;
      const [mx, my] = d3.pointer(evt, cadre);
      infobulle.style("left", (mx + 16) + "px").style("top", (my + 16) + "px");
    })
    .on("mouseleave", () => {
      if (infobullePinglee) return;
      infobulle.style("opacity", 0);
    })
    .on("click", (evt, d) => {
      evt.stopPropagation();
      const [mx, my] = d3.pointer(evt, cadre);
      infobullePinglee = true;
      infobulle
        .classed("epingle", true)
        .style("left", (mx + 16) + "px")
        .style("top", (my + 16) + "px")
        .style("opacity", 1)
        .html(contenuInfobulle(d) + `<div class="fermer-infobulle">Cliquer ailleurs pour fermer</div>`);
    });
}

dessiner();
window.addEventListener("resize", dessiner);

svg.call(d3.zoom().scaleExtent([0.3, 12]).on("zoom", (evt) => {
  g.attr("transform", evt.transform);
}));

// Legende unifiee : une entree par combinaison (source x theme) reellement
// presente dans les donnees, avec la forme utilisee dans le graphe (cercle
// Bible / triangle Ovide) et la couleur du theme. Clic = isoler ce groupe.
const LABEL_THEME = Object.fromEntries(THEMES.map(t => [t.cle, t.label]));
const LABEL_SOURCE = { bible: "Bible", ovide: "Ovide" };

const groupesLegende = {};
DONNEES.noeuds.forEach(n => {
  const cle = n.source + "|" + n.theme;
  if (!groupesLegende[cle]) groupesLegende[cle] = { source: n.source, theme: n.theme };
});
const combosActifs = new Set(Object.keys(groupesLegende));

const legende = d3.select("#legende");
["bible", "ovide"].forEach((source, i) => {
  const items = Object.values(groupesLegende).filter(g => g.source === source);
  if (!items.length) return;
  if (i > 0) legende.append("div").attr("class", "legende-sep");
  legende.append("h4").text(LABEL_SOURCE[source]);
  const sel = legende.selectAll(null)
    .data(items)
    .join("div")
    .attr("class", "legende-item")
    .on("click", (evt, d) => {
      const cle = d.source + "|" + d.theme;
      if (combosActifs.has(cle)) combosActifs.delete(cle); else combosActifs.add(cle);
      appliquerFiltre();
    });
  sel.append("svg").attr("class", "legende-forme").attr("viewBox", "-4 -4 8 8")
    .append("path")
    .attr("d", d => formeParSource[d.source]())
    .attr("fill", d => couleurTheme[d.theme]);
  sel.append("span").text(d => LABEL_THEME[d.theme] || d.theme);
});

function appliquerFiltre() {
  legende.selectAll(".legende-item").classed("inactif", d => !combosActifs.has(d.source + "|" + d.theme));
  gNoeuds.selectAll("path")
    .classed("estompe", d => !combosActifs.has(d.source + "|" + d.theme));
}

// Panneaux de statistiques par theme
const tableStats = d3.select("#table-stats");
tableStats.html(`
  <thead><tr><th>Theme (Ovide)</th><th>n</th><th>% meme theme cote Bible</th><th>Similarite moyenne</th></tr></thead>
  <tbody>
    ${DONNEES.stats.map(s => `
      <tr>
        <td>${LABEL_THEME[s.theme] || s.theme}</td>
        <td class="chiffre">${s.n}</td>
        <td class="chiffre">${(s.pct_meme_theme * 100).toFixed(0)}%</td>
        <td class="chiffre">${s.similarite_moyenne.toFixed(3)}</td>
      </tr>`).join("")}
  </tbody>
`);

const tableStatsBible = d3.select("#table-stats-bible");
tableStatsBible.html(`
  <thead><tr><th>Theme (Bible)</th><th>n</th><th>% meme theme cote Ovide</th><th>Similarite moyenne</th></tr></thead>
  <tbody>
    ${DONNEES.stats_bible.map(s => `
      <tr>
        <td>${LABEL_THEME[s.theme] || s.theme}</td>
        <td class="chiffre">${s.n}</td>
        <td class="chiffre">${(s.pct_meme_theme * 100).toFixed(0)}%</td>
        <td class="chiffre">${s.similarite_moyenne.toFixed(3)}</td>
      </tr>`).join("")}
  </tbody>
`);

// Bascule entre liens de voisinage local (tous types de paires) et les deux
// sens de correspondance (Ovide -> meilleure Bible / Bible -> meilleure Ovide).
function majAide() {
  let texte;
  if (modeLiens === "local") {
    texte = `Liens : ${DONNEES.liens.length} paires dont la similarite depasse le seuil retenu, calculee sur l'embedding 768D d'origine - toutes combinaisons Bible/Ovide confondues.`;
  } else if (modeLiens === "correspondance") {
    texte = `Liens : ${DONNEES.correspondances.length} correspondances Ovide -> meilleure Bible (une par illustration Ovide) - trait plein = meme theme des deux cotes, pointille = theme different.`;
  } else {
    texte = `Liens : ${DONNEES.correspondances_bible.length} correspondances Bible -> meilleure Ovide (une par illustration Bible) - trait plein = meme theme des deux cotes, pointille = theme different.`;
  }
  d3.select("#aide-liens").text(texte);
}

function dessinerLiens() {
  gLiens.selectAll("line").remove();
  if (modeLiens === "local") {
    gLiens.selectAll("line")
      .data(DONNEES.liens)
      .join("line")
      .attr("class", "lien")
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.source).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.source).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.target).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.target).y))
      .attr("stroke-width", 1)
      .attr("stroke-opacity", d => 0.05 + 0.35 * (d.similarite - simLocal[0]) / (simLocal[1] - simLocal[0] || 1));
  } else if (modeLiens === "correspondance") {
    gLiens.selectAll("line")
      .data(DONNEES.correspondances)
      .join("line")
      .attr("class", d => "lien lien-correspondance " + (d.meme_theme ? "memeTheme" : "autreTheme"))
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.ovide_id).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.ovide_id).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.bible_id).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.bible_id).y))
      .attr("stroke-width", d => d.meme_theme ? 1.4 : 1)
      .attr("stroke-opacity", d => 0.2 + 0.6 * (d.similarite - simCorr[0]) / (simCorr[1] - simCorr[0] || 1));
  } else {
    gLiens.selectAll("line")
      .data(DONNEES.correspondances_bible)
      .join("line")
      .attr("class", d => "lien lien-correspondance " + (d.meme_theme ? "memeTheme" : "autreTheme"))
      .attr("x1", d => xEchelleActuelle(parIdIndex.get(d.bible_id).x))
      .attr("y1", d => yEchelleActuelle(parIdIndex.get(d.bible_id).y))
      .attr("x2", d => xEchelleActuelle(parIdIndex.get(d.ovide_id).x))
      .attr("y2", d => yEchelleActuelle(parIdIndex.get(d.ovide_id).y))
      .attr("stroke-width", d => d.meme_theme ? 1.4 : 1)
      .attr("stroke-opacity", d => 0.2 + 0.6 * (d.similarite - simCorrBible[0]) / (simCorrBible[1] - simCorrBible[0] || 1));
  }
  majAide();
}

d3.select("#bascule").selectAll("button").on("click", function (evt, d) {
  modeLiens = d3.select(this).attr("data-mode");
  d3.select("#bascule").selectAll("button").classed("actif", false);
  d3.select(this).classed("actif", true);
  dessinerLiens();
});
</script>
</body>
</html>"""

html_final = (
    HTML_TEMPLATE
    .replace("__DONNEES_JSON__", DONNEES_JSON)
    .replace("__N_NOEUDS__", str(len(noeuds_json)))
)

DOSSIER_DATAVIS = RACINE / "resultats" / "Datavis"
DOSSIER_DATAVIS.mkdir(parents=True, exist_ok=True)
chemin_sortie_viz = DOSSIER_DATAVIS / "proximite_bibles_ovide.html"
chemin_sortie_viz.write_text(html_final, encoding="utf-8")
print(f"Visualisation generee : {chemin_sortie_viz}")
print(f"Taille : {chemin_sortie_viz.stat().st_size / 1024:.0f} Ko")

Visualisation generee : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/Datavis/proximite_bibles_ovide.html
Taille : 264 Ko


### Vérification — la position 2D reflète-t-elle vraiment la proximité 768D ?

UMAP est construit pour préserver les voisinages locaux (contrairement à une
disposition aléatoire ou à un simple hasard de mise en page), mais on le mesure
plutôt que de le supposer :
- **Trustworthiness** (`sklearn.manifold.trustworthiness`) : parmi les k plus
  proches voisins de chaque point dans la projection 2D, quelle proportion sont
  aussi ses k plus proches voisins dans l'espace d'origine (768D) ? 1.0 = parfait,
  0.5 = pas mieux qu'aléatoire.
- **Longueur des liens** : les liens tracés relient déjà les vraies plus
  proches voisines (768D — nombre exact affiché juste au-dessus). On compare
  leur longueur en 2D à celle de paires aléatoires — si la projection est
  fidèle, les liens doivent être nettement plus courts que des paires prises
  au hasard.

In [33]:
from sklearn.manifold import trustworthiness

score_confiance = trustworthiness(X_points, coords, n_neighbors=K_VOISINS, metric="cosine")
print(f"Trustworthiness (k={K_VOISINS}) : {score_confiance:.3f}  (1.0 = parfait, 0.5 = aléatoire)")

# distances 2D des vraies paires plus-proches-voisines (les liens tracés)
dist_liens = np.array([
    np.linalg.norm([points.loc[l['source'], 'x'] - points.loc[l['target'], 'x'],
                     points.loc[l['source'], 'y'] - points.loc[l['target'], 'y']])
    for l in liens
])

# distances 2D de paires aléatoires (référence "si c'était sans rapport")
rng = np.random.default_rng(42)
n_ref = 2000
idx_a = rng.integers(0, len(points), n_ref)
idx_b = rng.integers(0, len(points), n_ref)
dist_aleatoire = np.array([
    np.linalg.norm([points.loc[a, 'x'] - points.loc[b, 'x'], points.loc[a, 'y'] - points.loc[b, 'y']])
    for a, b in zip(idx_a, idx_b) if a != b
])

print(f"\nDistance 2D moyenne — vraies plus proches voisines (768D) : {dist_liens.mean():.2f}")
print(f"Distance 2D moyenne — paires aléatoires                    : {dist_aleatoire.mean():.2f}")
print(f"Ratio (plus petit = mieux) : {dist_liens.mean() / dist_aleatoire.mean():.2f}")

Trustworthiness (k=3) : 0.931  (1.0 = parfait, 0.5 = aléatoire)

Distance 2D moyenne — vraies plus proches voisines (768D) : 0.29
Distance 2D moyenne — paires aléatoires                    : 4.25
Ratio (plus petit = mieux) : 0.07


## Base vectorielle complete — 28 editions segmentees, metadonnees Synthese

Objectif : etendre la base vectorielle Ovide au-dela des 73 illustrations deja
themees (construites ci-dessus, en memoire, a partir de 3 feuilles de
`BNU_corpus.ods`). Ici on vectorise **toutes** les
illustrations deja segmentees dans `data/editions_ovide/segmentees/` (2191 crops,
28 dossiers/editions), chacune enrichie avec les metadonnees d'edition trouvees
dans la feuille **Synthese** de `BNU_corpus.ods` (titre, ville, graveur, technique,
annee, langue, famille iconographique).

**Ce que ça n'est pas** : un theme (deluge / creation_monde / ...) precis par
illustration — sauf pour les 73 deja connues, reprises telles quelles ici. Les
~2100 autres n'ont pas de theme individuel (voir feuilles `#N` de `BNU_corpus.ods`
pour une etape ulterieure, qui donnera un theme precis pour les editions ayant une
feuille dediee — Salomon, Wickram, Solis, Savery notamment).

In [ ]:
import unicodedata  # normalisation accents/casse pour le rattachement des theme (voir plus bas)


### 1. Rapprochement dossier segmente -> ligne Synthese

Deux voies : ark deja connu (documente dans les notebooks
`classification_bois_cuivre` / `classification_graveur`), ou a defaut un
rapprochement par ville + annee + fragment du nom du graveur (verifie a l'oeil,
voir le tableau affiche plus bas — 27/27 dossiers rapproches avec succes).

In [35]:
SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

# ark deja connus avec certitude (repris de vector_base.ipynb / classification_*/01_*.ipynb)
ARK_CONNU = {
    'bois_salomon_rouille_lyon1557': 'btv1b2200047r',
    'bois_wickram_behem_mayence1545': 'bsb10139926',
    'bois_solis_feyerabend_francfort1581': 'bsb00087854',
    'cuivre_savery_farnaby_paris1637': 'bsb10863401',
    'bois_eskrich_rouille_lyon1556': 'btv1b22000559',
    'bois_leroy_gueynard_lyon1510': 'bsb11054210',
    'cuivre_baur_sn_augsbourg1709': 'bsb10872075',
    'cuivre_baur_sn_vienne1639': 'bsb10872073',
    'cuivre_borcht_plantin_anvers1591': 'bsb00004340',
    'cuivre_bouche_blaeu_amsterdam1702': 'bpt6k15151988',
    'cuivre_depasse_depasse_koln1602': 'bpt6k15218623',
    'cuivre_depasse_jansonius_arnhem1607': 'bpt6k1522448r',
    'cuivre_gaultier_guillemot_paris1610': 'bsb11913284',
    'cuivre_gaultier_veuveguillemot_paris1614': 'bpt6k6277348n',
    'cuivre_goltzius_goltzius_haarlem1589': 'btv1b10534945n',
    'cuivre_goltzius_goltzius_haarlem1589_couleur': 'btv1b10534945n',
    'cuivre_isaac_langelier_paris1617': 'bpt6k722055',
    'cuivre_mathieu_langelier_paris1619': 'btv1b22000826',
    'cuivre_monconet_sommaville_paris1660': 'bpt6k87045023',
    'cuivre_tempesta_dejode_anvers1606': 'btv1b54000051z',
    'cuivre_tempesta_jansonius_amsterdam1610': 'bsb00008186',
}

PAT_ARK = re.compile(r'ark:/12148/([a-zA-Z0-9]+)|(bsb\d+)')


def extraire_ark(row):
    for col in ['version numérisée 1', 'version numérisée 2', 'url catalogue']:
        val = str(row.get(col))
        m = PAT_ARK.search(val)
        if m:
            return m.group(1) or m.group(2)
    return None


def normaliser(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())


xl = pd.ExcelFile(RACINE / "retours_celine" / "BNU_corpus.ods", engine='odf')
synthese = xl.parse('Synthèse', header=0)
synthese['ark_num'] = synthese.apply(extraire_ark, axis=1)
synthese['graveur_norm'] = synthese['graveur\xa0: Nom, Prénom'].apply(normaliser)
synthese['ville_norm'] = synthese['ville'].apply(normaliser)

dossiers = sorted(d.name for d in SEG_DIR.iterdir() if d.is_dir() and d.name != "bnu_corpus_celine")
print(f"{len(dossiers)} dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)")

27 dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)


In [36]:
def rapprocher(dossier):
    m = re.match(r"^(bois|cuivre)_([a-z0-9]+)_([a-z0-9]+)_([a-z]+?)(\d{4})(?:_(.+))?$", dossier)
    if not m:
        return None, "NOM NON PARSE"
    technique, graveur_court, editeur_court, ville_court, annee, variante = m.groups()

    ark = ARK_CONNU.get(dossier)
    if ark:
        candidats = synthese[synthese["ark_num"] == ark]
        if len(candidats):
            return candidats.iloc[0], "ark connu -> trouve"

    prefixe = "ark connu mais absent de Synthese" if ark else "pas d'ark"
    candidats = synthese[
        (synthese["ville_norm"].str.contains(ville_court, na=False))
        & (synthese["année"].astype(str).str.contains(str(annee), na=False))
    ]
    if len(candidats) == 0:
        return None, f"{prefixe} -> AUCUN CANDIDAT"
    if len(candidats) == 1:
        return candidats.iloc[0], f"{prefixe} -> trouve par ville+annee"
    affine = candidats[candidats["graveur_norm"].str.contains(graveur_court[:5], na=False)]
    if len(affine):
        return affine.iloc[0], f"{prefixe} -> affine par graveur ({len(candidats)} candidats)"
    return None, f"{prefixe} -> AMBIGU ({len(candidats)} candidats)"


CHAMPS_SYNTHESE = {
    "titre": "titre abrégé", "ville": "ville", "publisher": "publisher", "annee": "année",
    "langue": "langue", "technique": "technique", "graveur": "graveur\xa0: Nom, Prénom",
    "type_iconographique": "type iconographique", "cadre_grave": "cadre gravé",
    "famille_iconographique": "familles iconographiques", "ark_synthese": "ark_num",
}

meta_par_dossier = {}
for dossier in dossiers:
    ligne, statut = rapprocher(dossier)
    rec = {"statut": statut}
    if ligne is not None:
        for champ, col in CHAMPS_SYNTHESE.items():
            rec[champ] = ligne[col]
    meta_par_dossier[dossier] = rec

table_meta = pd.DataFrame(meta_par_dossier).T
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
print(table_meta[["statut", "titre", "ville", "annee", "graveur", "famille_iconographique"]])
print()
print(table_meta["statut"].str.replace(r"\(.*\)", "", regex=True).value_counts())

                                                       statut                          titre                  ville      annee                        graveur famille_iconographique
bois_eskrich_rouille_lyon1556             ark connu -> trouve  Trois Premiers livres de l...                   Lyon       1556                Eskrich, Pierre                      7
bois_leroy_gueynard_lyon1510    ark connu mais absent de S...  P. Ovidii Nasonis Metamorp...                   Lyon       1510            Leroy II, Guillaume                      2
bois_salomon_rouille_lyon1557             ark connu -> trouve        La Metamorphose figurée                   Lyon       1557               Salomon, Bernard                      7
bois_solis_feyerabend_franc...            ark connu -> trouve  P. Ovidii Metamorphosis, O...  Francfort-sur-le-Main       1581                  Solis, Virgil                      7
bois_wickram_behem_mayence1545            ark connu -> trouve  P. Ovidii Nasonis dess all...   

### 2. Themes deja connus (73 illustrations, voir ci-dessus)

Reutilises tels quels — pas de re-classification, juste report par chemin de
fichier, pour ne pas perdre l'information deja validee.

In [37]:
# base_ovide (73 illustrations) construit plus haut dans ce meme notebook,
# jamais sauvegarde comme fichier separe -- reutilise directement en memoire.
THEME_CONNU = dict(zip(base_ovide["chemin"].apply(lambda c: Path(c).name), base_ovide["theme"]))
print(f"{len(THEME_CONNU)} themes deja connus (par nom de fichier)")

73 themes deja connus (par nom de fichier)


### 3. Vectorisation SigLIP (niveaux de gris) — tous les crops des 28 dossiers

Meme pretraitement que ci-dessus (niveaux de gris puis re-RGB, pour neutraliser
le coloriage) — la nouvelle base reste comparable
aux donnees existantes (`bibles_siglip.pkl`, et les 73 illustrations Ovide
deja themees construites plus haut).

In [ ]:
processor, model_siglip = charger_siglip(DEVICE)
print("SigLIP charge")


In [ ]:
import time

lignes = []
t0 = time.time()
total = 0
for dossier in dossiers + ["bnu_corpus_celine"]:
    meta = meta_par_dossier.get(dossier, {})
    fichiers = sorted((SEG_DIR / dossier).glob("*.jpg"))
    for k, chemin in enumerate(fichiers, 1):
        vec = embed_image(chemin, processor, model_siglip, DEVICE)
        lignes.append({
            "chemin": str(chemin),
            "dossier": dossier,
            "theme": THEME_CONNU.get(chemin.name),
            "titre": meta.get("titre"),
            "ville": meta.get("ville"),
            "publisher": meta.get("publisher"),
            "annee": meta.get("annee"),
            "langue": meta.get("langue"),
            "technique": meta.get("technique"),
            "graveur": meta.get("graveur"),
            "type_iconographique": meta.get("type_iconographique"),
            "cadre_grave": meta.get("cadre_grave"),
            "famille_iconographique": meta.get("famille_iconographique"),
            "ark": meta.get("ark_synthese"),
            "embedding": vec.tolist(),
        })
    total += len(fichiers)
    print(f"  {dossier:45s} {len(fichiers):4d} crops   (cumul {total}, {time.time()-t0:.0f}s)")

print(f"\nTermine : {len(lignes)} illustrations vectorisees en {time.time()-t0:.0f}s")

### 4. Sauvegarde

In [40]:
corpus_complet = pd.DataFrame(lignes)
print(corpus_complet.shape)
print(corpus_complet["theme"].notna().sum(), "illustrations avec theme connu")
print(corpus_complet.groupby("dossier").size().sort_values(ascending=False))

DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
chemin_sortie = DOSSIER_VECTOR_DB / "ovide_corpus_complet_siglip.pkl"
corpus_complet.to_pickle(chemin_sortie)
print(f"\nSauvegarde : {chemin_sortie}  ({chemin_sortie.stat().st_size / 1024:.0f} Ko)")

(2191, 15)
77 illustrations avec theme connu
dossier
bois_solis_feyerabend_francfort1581             184
cuivre_borcht_plantin_anvers1591                182
bois_salomon_rouille_lyon1557                   161
cuivre_baur_sn_augsbourg1709                    161
cuivre_tempesta_jansonius_amsterdam1610         149
cuivre_monconet_sommaville_paris1660            148
cuivre_tempesta_dejode_anvers1606               139
cuivre_depasse_jansonius_arnhem1607             136
cuivre_mathieu_langelier_paris1619              135
cuivre_depasse_depasse_koln1602                 134
cuivre_bouche_blaeu_amsterdam1702               126
cuivre_baur_sn_vienne1639                       125
bnu_corpus_celine                                62
bois_wickram_behem_mayence1545                   50
bois_eskrich_rouille_lyon1556                    42
cuivre_goltzius_goltzius_haarlem1589_couleur     38
cuivre_goltzius_goltzius_haarlem1589             38
cuivre_briot_drobet_lyon1628                     28
bois_leroy_


Sauvegarde : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/ovide_corpus_complet_siglip.pkl  (15278 Ko)


## Bilan

Base vectorielle complete du corpus Ovide segmente : chaque illustration porte les
metadonnees d'edition (titre, ville, graveur, technique, annee, langue, famille
iconographique) meme sans theme individuel connu. Les 73 illustrations deja
themees ci-dessus gardent leur theme.

**Étape suivante, dans `themes_precis.ipynb`** : pour les editions avec
feuille `#N` dediee dans `BNU_corpus.ods` (Salomon -> `#0_CORPUS_REF`, Wickram ->
`#1`, Solis -> `#7`, Savery -> `#11`), extraire le theme precis de chaque planche
en calant folio <-> page scannee (voir section "Themes precis" ci-dessous).